# Phase 5: Gold Layer – KPIs, Aggregations & Time Travel

This notebook builds business‑ready aggregated tables from the Silver layer
and demonstrates Delta Lake's time‑travel capability.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Phase5_GoldLayer") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print("Spark ready for Gold layer.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 14:32:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark ready for Gold layer.


## 1. Load Silver Data

Read the cleaned Silver table from MinIO.

In [2]:
silver_path = "s3a://ecommerce-silver/delta/silver/online_retail"
df_silver = spark.read.format("delta").load(silver_path)

print(f"Silver rows: {df_silver.count():,}")
df_silver.printSchema()

26/06/12 14:33:06 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/06/12 14:33:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 3:===============================================>         (42 + 1) / 50]

Silver rows: 534,129
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- InvoiceDate_clean: timestamp (nullable = true)



In [4]:
from pyspark.sql.functions import col

# Calculate revenue: Quantity * UnitPrice
df_silver = df_silver.withColumn("Revenue", col("Quantity") * col("UnitPrice"))

print("Revenue column added.")
df_silver.select("Quantity", "UnitPrice", "Revenue").show(5, truncate=False)

Revenue column added.


+--------+---------+------------------+
|Quantity|UnitPrice|Revenue           |
+--------+---------+------------------+
|48      |2.1      |100.80000000000001|
|5       |2.1      |10.5              |
|1       |1.65     |1.65              |
|6       |1.65     |9.899999999999999 |
|12      |1.25     |15.0              |
+--------+---------+------------------+
only showing top 5 rows



## 2. Daily Sales Aggregation

Total revenue per day based on `InvoiceDate_clean`.

In [5]:
from pyspark.sql.functions import to_date, sum as _sum, countDistinct

daily_sales = df_silver.groupBy(to_date(col("InvoiceDate_clean")).alias("SaleDate")) \
    .agg(
        _sum("Revenue").alias("TotalRevenue"),
        countDistinct("InvoiceNo").alias("NumberOfInvoices")
    ) \
    .orderBy("SaleDate")

print("Daily sales sample:")
daily_sales.show(10, truncate=False)

Daily sales sample:


[Stage 17:=============================>                            (1 + 1) / 2]

+----------+------------------+----------------+
|SaleDate  |TotalRevenue      |NumberOfInvoices|
+----------+------------------+----------------+
|2010-12-01|58451.560000000005|133             |
|2010-12-02|46088.31999999997 |165             |
|2010-12-03|45575.38000000002 |75              |
|2010-12-05|30973.63000000001 |95              |
|2010-12-06|53653.87000000004 |120             |
|2010-12-07|44994.7           |102             |
|2010-12-08|44035.22000000002 |139             |
|2010-12-09|52494.14          |141             |
|2010-12-10|57303.01000000001 |84              |
|2010-12-12|17037.500000000004|51              |
+----------+------------------+----------------+
only showing top 10 rows



In [8]:
daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .save("s3a://ecommerce-gold/delta/gold/daily_sales")

print("Daily sales table saved to Gold layer.")

Daily sales table saved to Gold layer.


## 3. Monthly Sales Trend

In [9]:
from pyspark.sql.functions import date_format, month, year

monthly_sales = df_silver.groupBy(
        year(col("InvoiceDate_clean")).alias("Year"),
        month(col("InvoiceDate_clean")).alias("Month")
    ) \
    .agg(
        _sum("Revenue").alias("TotalRevenue"),
        countDistinct("InvoiceNo").alias("NumberOfInvoices")
    ) \
    .orderBy("Year", "Month")

monthly_sales.show(15, truncate=False)

+----+-----+------------------+----------------+
|Year|Month|TotalRevenue      |NumberOfInvoices|
+----+-----+------------------+----------------+
|2010|12   |746723.6099999994 |1885            |
|2011|1    |558448.559999999  |1346            |
|2011|2    |497026.41         |1319            |
|2011|3    |682013.9800000016 |1772            |
|2011|4    |492367.8410000002 |1486            |
|2011|5    |722094.099999999  |1995            |
|2011|6    |689977.2300000007 |1862            |
|2011|7    |680156.9910000013 |1745            |
|2011|8    |703510.5800000009 |1639            |
|2011|9    |1017596.6820000027|2170            |
|2011|10   |1069368.2299999986|2402            |
|2011|11   |1456145.799999999 |3210            |
|2011|12   |432701.06         |965             |
+----+-----+------------------+----------------+



In [10]:
monthly_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .save("s3a://ecommerce-gold/delta/gold/monthly_sales")

print("Monthly sales table saved to Gold layer.")

Monthly sales table saved to Gold layer.


## 4. Top‑Selling Products

By total revenue and by quantity sold.

In [11]:
top_products_revenue = df_silver.groupBy("StockCode", "Description") \
    .agg(_sum("Revenue").alias("TotalRevenue")) \
    .orderBy(col("TotalRevenue").desc())

print("Top 10 products by revenue:")
top_products_revenue.show(10, truncate=False)

Top 10 products by revenue:
+---------+----------------------------------+------------------+
|StockCode|Description                       |TotalRevenue      |
+---------+----------------------------------+------------------+
|DOT      |DOTCOM POSTAGE                    |206245.47999999992|
|22423    |REGENCY CAKESTAND 3 TIER          |164459.48999999982|
|47566    |PARTY BUNTING                     |98243.88000000038 |
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER|97659.93999999986 |
|85099B   |JUMBO BAG RED RETROSPOT           |92175.79000000056 |
|23084    |RABBIT NIGHT LIGHT                |66661.62999999989 |
|POST     |POSTAGE                           |66230.64000000001 |
|22086    |PAPER CHAIN KIT 50'S CHRISTMAS    |63715.24000000023 |
|84879    |ASSORTED COLOUR BIRD ORNAMENT     |58792.42000000036 |
|79321    |CHILLI LIGHTS                     |53746.65999999994 |
+---------+----------------------------------+------------------+
only showing top 10 rows



In [12]:
top_products_qty = df_silver.groupBy("StockCode", "Description") \
    .agg(_sum("Quantity").alias("TotalQuantity")) \
    .orderBy(col("TotalQuantity").desc())

print("Top 10 products by quantity sold:")
top_products_qty.show(10, truncate=False)

Top 10 products by quantity sold:
+---------+----------------------------------+-------------+
|StockCode|Description                       |TotalQuantity|
+---------+----------------------------------+-------------+
|84077    |WORLD WAR 2 GLIDERS ASSTD DESIGNS |53751        |
|85099B   |JUMBO BAG RED RETROSPOT           |47256        |
|22197    |POPCORN HOLDER                    |36322        |
|84879    |ASSORTED COLOUR BIRD ORNAMENT     |36282        |
|21212    |PACK OF 72 RETROSPOT CAKE CASES   |36016        |
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER|35002        |
|23084    |RABBIT NIGHT LIGHT                |30631        |
|22492    |MINI PAINT SET VINTAGE            |26437        |
|22616    |PACK OF 12 LONDON TISSUES         |26095        |
|21977    |PACK OF 60 PINK PAISLEY CAKE CASES|24719        |
+---------+----------------------------------+-------------+
only showing top 10 rows



In [13]:
top_products_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .save("s3a://ecommerce-gold/delta/gold/top_products_revenue")

print("Top products (by revenue) table saved to Gold layer.")

Top products (by revenue) table saved to Gold layer.


## 5. Sales by Country

In [14]:
sales_by_country = df_silver.groupBy("Country") \
    .agg(
        _sum("Revenue").alias("TotalRevenue"),
        countDistinct("InvoiceNo").alias("NumberOfInvoices")
    ) \
    .orderBy(col("TotalRevenue").desc())

sales_by_country.show(20, truncate=False)

+---------------+------------------+----------------+
|Country        |TotalRevenue      |NumberOfInvoices|
+---------------+------------------+----------------+
|United Kingdom |8189252.303999934 |21391           |
|Netherlands    |284661.54000000004|100             |
|EIRE           |262993.37999999995|360             |
|Germany        |221509.47000000003|603             |
|France         |197317.1100000001 |461             |
|Australia      |137009.77000000002|69              |
|Switzerland    |56363.05000000002 |74              |
|Spain          |54756.03000000001 |105             |
|Belgium        |40910.96000000001 |119             |
|Sweden         |36585.409999999996|46              |
|Japan          |35340.62000000001 |28              |
|Norway         |35163.46          |40              |
|Portugal       |29302.97          |71              |
|Finland        |22326.739999999994|48              |
|Channel Islands|20076.390000000007|33              |
|Denmark        |18768.14   

In [15]:
sales_by_country.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .save("s3a://ecommerce-gold/delta/gold/sales_by_country")

print("Sales by country table saved to Gold layer.")

Sales by country table saved to Gold layer.


## 6. Top Customers by Revenue

In [17]:
top_customers = df_silver.filter(col("CustomerID") != -1) \
    .groupBy("CustomerID") \
    .agg(
        _sum("Revenue").alias("TotalRevenue"),
        countDistinct("InvoiceNo").alias("NumberOfInvoices")
    ) \
    .orderBy(col("TotalRevenue").desc())

print("Top 10 customers by revenue:")
top_customers.show(10, truncate=False)

Top 10 customers by revenue:
+----------+------------------+----------------+
|CustomerID|TotalRevenue      |NumberOfInvoices|
+----------+------------------+----------------+
|14646     |279489.02         |76              |
|18102     |256438.49000000005|62              |
|17450     |187322.16999999995|55              |
|14911     |132458.73000000007|248             |
|12415     |123725.45000000004|26              |
|14156     |113214.59000000003|66              |
|17511     |88125.38          |46              |
|16684     |65892.08          |31              |
|13694     |62690.53999999998 |60              |
|15311     |59284.189999999995|118             |
+----------+------------------+----------------+
only showing top 10 rows



In [18]:
top_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .save("s3a://ecommerce-gold/delta/gold/top_customers")

print("Top customers table saved to Gold layer.")

Top customers table saved to Gold layer.


## 7. Time Travel Demo

We'll update a specific record, then query the table as it was before the update.

In [19]:
# Pick a specific invoice item for demonstration
sample_row = df_silver.filter(col("InvoiceNo") == "536365") \
    .filter(col("StockCode") == "85123A") \
    .select("InvoiceNo", "StockCode", "Quantity", "UnitPrice", "Revenue")

sample_row.show(truncate=False)

# Save original values for later comparison
original_qty = sample_row.collect()[0]["Quantity"]
original_price = sample_row.collect()[0]["UnitPrice"]
original_revenue = sample_row.collect()[0]["Revenue"]

print(f"Original: Qty={original_qty}, Price={original_price}, Revenue={original_revenue}")

+---------+---------+--------+---------+------------------+
|InvoiceNo|StockCode|Quantity|UnitPrice|Revenue           |
+---------+---------+--------+---------+------------------+
|536365   |85123A   |6       |2.55     |15.299999999999999|
+---------+---------+--------+---------+------------------+

Original: Qty=6, Price=2.55, Revenue=15.299999999999999


In [22]:
# Create a temporary Delta table from Silver for the time-travel demo
demo_path = "s3a://ecommerce-gold/delta/gold/time_travel_demo"
df_silver.write.format("delta").mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .save(demo_path)

# Read back the demo table and register as a temp view
df_demo = spark.read.format("delta").load(demo_path)
df_demo.createOrReplaceTempView("demo_table")

print("Demo table created and registered as 'demo_table'.")

Demo table created and registered as 'demo_table'.


In [23]:
import time
before_update_ts = spark.sql("SELECT current_timestamp()").collect()[0][0]
print("Timestamp before update:", before_update_ts)

Timestamp before update: 2026-06-12 14:47:01.314781


In [24]:
# Update: double the unit price of that specific row
update_query = """
UPDATE demo_table
SET UnitPrice = UnitPrice * 2
WHERE InvoiceNo = '536365' AND StockCode = '85123A'
"""
spark.sql(update_query)
print("UPDATE executed.")

UPDATE executed.


In [25]:
spark.sql("""
    SELECT InvoiceNo, StockCode, Quantity, UnitPrice, Revenue
    FROM demo_table
    WHERE InvoiceNo = '536365' AND StockCode = '85123A'
""").show(truncate=False)

+---------+---------+--------+---------+------------------+
|InvoiceNo|StockCode|Quantity|UnitPrice|Revenue           |
+---------+---------+--------+---------+------------------+
|536365   |85123A   |6       |5.1      |15.299999999999999|
+---------+---------+--------+---------+------------------+



### Travel back to before the update

We can use either a version number or a timestamp.

In [26]:
# Find the previous version (usually version 0 is initial write, version 1 is the update)
# We can use VERSION AS OF 0 to get the state before the update.
spark.sql(f"""
    SELECT InvoiceNo, StockCode, Quantity, UnitPrice, Revenue
    FROM delta.`{demo_path}` VERSION AS OF 0
    WHERE InvoiceNo = '536365' AND StockCode = '85123A'
""").show(truncate=False)

[Stage 193:=========================================>             (38 + 1) / 50]

+---------+---------+--------+---------+------------------+
|InvoiceNo|StockCode|Quantity|UnitPrice|Revenue           |
+---------+---------+--------+---------+------------------+
|536365   |85123A   |6       |2.55     |15.299999999999999|
+---------+---------+--------+---------+------------------+



In [27]:
spark.sql(f"""
    SELECT InvoiceNo, StockCode, Quantity, UnitPrice, Revenue
    FROM delta.`{demo_path}` TIMESTAMP AS OF '{before_update_ts}'
    WHERE InvoiceNo = '536365' AND StockCode = '85123A'
""").show(truncate=False)

[Stage 198:================================================>      (44 + 1) / 50]

+---------+---------+--------+---------+------------------+
|InvoiceNo|StockCode|Quantity|UnitPrice|Revenue           |
+---------+---------+--------+---------+------------------+
|536365   |85123A   |6       |2.55     |15.299999999999999|
+---------+---------+--------+---------+------------------+



## 8. Done

Gold tables are stored in `ecommerce-gold` bucket. Time‑travel capability verified.

In [28]:
spark.stop()